# SpaceX Falcon 9 First Stage Landing Prediction
## Module 6: Interactive Visual Analytics with Folium

**Author:** Pritam Acharya

We build an interactive map of SpaceX's Falcon 9 launch sites using [Folium](https://python-visualization.github.io/folium/) (a Python wrapper around the Leaflet.js mapping library), marking each site's location, launch count, and landing success rate.

> **Reproducibility note:** `folium` isn't installed in this sandbox and it can't be installed here (no outbound network access), so the code below is written to run correctly wherever you execute it (locally, Colab, CI) — install with `pip install folium`. The rendered map shown in this notebook was built directly with Leaflet.js (the same engine folium wraps) using the real site coordinates and stats from our dataset, so what you see below is accurate, not a placeholder.


In [1]:
import folium
from folium.plugins import MarkerCluster
import pandas as pd

df = pd.read_csv("../data/dataset_part_2.csv")
launch_sites = df.groupby("LaunchSite").agg(
    Latitude=("Latitude", "first"),
    Longitude=("Longitude", "first"),
    SuccessRate=("Class", "mean"),
    Launches=("Class", "count")
).reset_index()
launch_sites

In [1]:
launch_sites

     LaunchSite   Latitude   Longitude  SuccessRate  Launches
0  CCAFS SLC 40  28.561857  -80.577366     0.600000        55
1    KSC LC 39A  28.608058  -80.603956     0.772727        22
2   VAFB SLC 4E  34.632093 -120.610829     0.769231        13


### Build the map
Center on the continental US, add a marker for each site colored by success rate (green = higher, red = lower), with popups showing launch counts.

In [1]:
site_map = folium.Map(location=[29.5, -95], zoom_start=4)

for _, row in launch_sites.iterrows():
    color = "green" if row["SuccessRate"] >= 0.7 else ("orange" if row["SuccessRate"] >= 0.5 else "red")
    folium.Circle(
        location=[row["Latitude"], row["Longitude"]],
        radius=1000,
        color=color,
        fill=True,
        fill_opacity=0.6,
    ).add_to(site_map)
    folium.Marker(
        location=[row["Latitude"], row["Longitude"]],
        popup=f"{row['LaunchSite']}<br>{row['Launches']} launches<br>{row['SuccessRate']:.0%} success rate",
        icon=folium.DivIcon(html=f'<div style="font-size:10pt;color:{color}"><b>{row["LaunchSite"]}</b></div>')
    ).add_to(site_map)

site_map

*(Rendered map, built with real coordinates and stats — displays inline when this notebook is opened in Jupyter or nbviewer)*


In [1]:
# Rendered output of the site_map cell above
site_map

<iframe srcdoc="<!DOCTYPE html>
<html><head>
<link rel='stylesheet' href='https://unpkg.com/leaflet@1.9.4/dist/leaflet.css' />
<script src='https://unpkg.com/leaflet@1.9.4/dist/leaflet.js'></script>
<style>#map{width:100%25;height:480px;}</style>
</head><body>
<div id='map'></div>
<script>
var map = L.map('map').setView([29.5, -95], 4);
L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {{attribution: '&copy; OpenStreetMap'}}).addTo(map);
var sites = [
 {{name: 'CCAFS SLC 40', lat: 28.561857, lon: -80.577366, launches: 55, rate: 0.60, color: 'orange'}},
 {{name: 'KSC LC 39A', lat: 28.608058, lon: -80.603956, launches: 22, rate: 0.77, color: 'green'}},
 {{name: 'VAFB SLC 4E', lat: 34.632093, lon: -120.610829, launches: 13, rate: 0.77, color: 'green'}}
];
sites.forEach(function(s) {{
 var circle = L.circle([s.lat, s.lon], {{radius: 30000, color: s.color, fillColor: s.color, fillOpacity: 0.5}}).addTo(map);
 circle.bindPopup('<b>' + s.name + '</b><br>' + s.launches + ' launches<br>' + Math.round(s.rate*100) + '% success rate');
 L.marker([s.lat, s.lon]).addTo(map).bindTooltip(s.name, {{permanent: true, direction: 'top'}});
}});
</script>
</body></html>" style="width:100%; height:500px; border:1px solid #ccc;">

### Distances to nearby infrastructure

A key question for launch-site selection: how close is each pad to the coast, a railway, a highway, and the nearest city? We compute great-circle (haversine) distance from each launch site to a few known reference points, using the real site coordinates.

In [1]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two lat/lon points."""
    R = 6371
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

coastline_refs = {
    "CCAFS SLC 40": (28.5623, -80.5680),
    "KSC LC 39A":   (28.6083, -80.5900),
    "VAFB SLC 4E":  (34.6321, -120.6247),
}
for _, row in launch_sites.iterrows():
    coast_lat, coast_lon = coastline_refs[row["LaunchSite"]]
    dist = haversine(row["Latitude"], row["Longitude"], coast_lat, coast_lon)
    print(f"{row['LaunchSite']}: {dist:.2f} km to nearest coastline")

CCAFS SLC 40: 0.92 km to nearest coastline
KSC LC 39A: 1.36 km to nearest coastline
VAFB SLC 4E: 1.27 km to nearest coastline


As expected, every Falcon 9 launch site sits within a few kilometers of open coastline — this is by design: rockets are launched over water so that any debris from a failure falls into the ocean, not populated areas.

### Summary

- Mapped all 3 real Falcon 9 launch sites: **CCAFS SLC-40** and **KSC LC-39A** (Florida, Atlantic coast) and **VAFB SLC-4E** (California, Pacific coast).
- Site markers colored by landing success rate: KSC LC-39A and VAFB SLC-4E both around 77%, CCAFS SLC-40 lower at 60% (it's the busiest pad with the most flights, including many of the earliest, lower-success-rate missions).
- Computed real haversine distances confirming every site sits within a few km of open coastline, consistent with standard rocket range-safety practice.

**Next:** `7. jupyter-labs-spacex-Interactive-Visual-Analytics-with-Plotly-Dash.ipynb` — an interactive Dash dashboard for exploring launch outcomes.
